In [1]:
import os
import gc
import re
import numpy as np
import pandas as pd

# Optional plotting libs; keep but don't require runtime usage
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

import tensorflow as tf
from keras.preprocessing.sequence import pad_sequences
from keras.layers import (
    Dense,
    Dropout,
    Embedding,
    LSTM,
    Bidirectional,
    Input,
    Concatenate,
    GRU,
    Lambda,
    Dot,
    Activation,
    Average,
)
from keras.models import Model

# Fix: ensure required NLTK resources exist in Kaggle offline environment
# (punkt_tab is used by some newer nltk versions; try both safely)
for pkg in ["stopwords", "punkt"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg == "stopwords" else f"tokenizers/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    # Not always available; ignore if download fails
    try:
        nltk.download("punkt_tab", quiet=True)
    except Exception:
        pass

eng_stopwords = [
    "i",
    "me",
    "my",
    "myself",
    "we",
    "our",
    "ours",
    "ourselves",
    "you",
    "you're",
    "you've",
    "you'll",
    "you'd",
    "your",
    "yours",
    "yourself",
    "yourselves",
    "he",
    "him",
    "his",
    "himself",
    "she",
    "she's",
    "her",
    "hers",
    "herself",
    "it",
    "it's",
    "its",
    "itself",
    "they",
    "them",
    "their",
    "theirs",
    "themselves",
    "what",
    "which",
    "who",
    "whom",
    "this",
    "that",
    "that'll",
    "these",
    "those",
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "have",
    "has",
    "had",
    "having",
    "do",
    "does",
    "did",
    "doing",
    "a",
    "an",
    "the",
    "and",
    "but",
    "if",
    "or",
    "because",
    "as",
    "until",
    "while",
    "of",
    "at",
    "by",
    "for",
    "with",
    "about",
    "against",
    "between",
    "into",
    "through",
    "during",
    "before",
    "after",
    "above",
    "below",
    "to",
    "from",
    "up",
    "down",
    "in",
    "out",
    "on",
    "off",
    "over",
    "under",
    "again",
    "further",
    "then",
    "once",
    "here",
    "there",
    "when",
    "where",
    "why",
    "how",
    "all",
    "any",
    "both",
    "each",
    "few",
    "more",
    "most",
    "other",
    "some",
    "such",
    "no",
    "nor",
    "not",
    "only",
    "own",
    "same",
    "so",
    "than",
    "too",
    "very",
    "s",
    "t",
    "can",
    "will",
    "just",
    "don",
    "don't",
    "should",
    "should've",
    "now",
    "d",
    "ll",
    "m",
    "o",
    "re",
    "ve",
    "y",
    "ain",
    "aren",
    "aren't",
    "couldn",
    "couldn't",
    "didn",
    "didn't",
    "doesn",
    "doesn't",
    "hadn",
    "hadn't",
    "hasn",
    "hasn't",
    "haven",
    "haven't",
    "isn",
    "isn't",
    "ma",
    "mightn",
    "mightn't",
    "mustn",
    "mustn't",
    "needn",
    "needn't",
    "shan",
    "shan't",
    "shouldn",
    "shouldn't",
    "wasn",
    "wasn't",
    "weren",
    "weren't",
    "won",
    "won't",
    "wouldn",
    "wouldn't",
]

puncts = [
    ",",
    ".",
    '"',
    ":",
    ")",
    "(",
    "-",
    "!",
    "?",
    "|",
    ";",
    "'",
    "$",
    "&",
    "/",
    "[",
    "]",
    ">",
    "%",
    "=",
    "#",
    "*",
    "+",
    "\\",
    "•",
    "~",
    "@",
    "£",
    "·",
    "_",
    "{",
    "}",
    "©",
    "^",
    "®",
    "`",
    "<",
    "→",
    "°",
    "€",
    "™",
    "›",
    "♥",
    "←",
    "×",
    "§",
    "″",
    "′",
    "Â",
    "█",
    "½",
    "à",
    "…",
    "\xa0",
    "\t",
    "“",
    "★",
    "”",
    "–",
    "●",
    "â",
    "►",
    "−",
    "¢",
    "²",
    "¬",
    "░",
    "¶",
    "↑",
    "±",
    "¿",
    "▾",
    "═",
    "¦",
    "║",
    "―",
    "¥",
    "▓",
    "—",
    "‹",
    "─",
    "\u3000",
    "\u202f",
    "▒",
    "：",
    "¼",
    "⊕",
    "▼",
    "▪",
    "†",
    "■",
    "’",
    "▀",
    "¨",
    "▄",
    "♫",
    "☆",
    "é",
    "¯",
    "♦",
    "¤",
    "▲",
    "è",
    "¸",
    "¾",
    "Ã",
    "⋅",
    "‘",
    "∞",
    "«",
    "∙",
    "）",
    "↓",
    "、",
    "│",
    "（",
    "»",
    "，",
    "♪",
    "╩",
    "╚",
    "³",
    "・",
    "╦",
    "╣",
    "╔",
    "╗",
    "▬",
    "❤",
    "ï",
    "Ø",
    "¹",
    "≤",
    "‡",
    "√",
]

mispell_dict = {
    "aren't": "are not",
    "can't": "cannot",
    "couldn't": "could not",
    "couldnt": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "doesnt": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "havent": "have not",
    "he'd": "he would",
    "he'll": "he will",
    "he's": "he is",
    "i'd": "I would",
    "i'd": "I had",
    "i'll": "I will",
    "i'm": "I am",
    "isn't": "is not",
    "it's": "it is",
    "it'll": "it will",
    "i've": "I have",
    "let's": "let us",
    "mightn't": "might not",
    "mustn't": "must not",
    "shan't": "shall not",
    "she'd": "she would",
    "she'll": "she will",
    "she's": "she is",
    "shouldn't": "should not",
    "shouldnt": "should not",
    "that's": "that is",
    "thats": "that is",
    "there's": "there is",
    "theres": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "theyre": "they are",
    "they've": "they have",
    "we'd": "we would",
    "we're": "we are",
    "weren't": "were not",
    "we've": "we have",
    "what'll": "what will",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "where's": "where is",
    "who'd": "who would",
    "who'll": "who will",
    "who're": "who are",
    "who's": "who is",
    "who've": "who have",
    "won't": "will not",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are",
    "you've": "you have",
    "'re": " are",
    "wasn't": "was not",
    "we'll": " will",
    "didn't": "did not",
    "tryin'": "trying",
}


def clean_text(text):
    # Fix: robust to NaNs/non-strings
    if text is None or (isinstance(text, float) and np.isnan(text)):
        text = ""
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9^,!.\/'+-=]", " ", text)
    text = text.lower().split()
    # Fix: use NLTK stopwords safely (downloaded above)
    stops = set(stopwords.words("english"))
    text = [w for w in text if w not in stops]
    return " ".join(text)


def _get_mispell(mispell_dict):
    mispell_re = re.compile("(%s)" % "|".join(map(re.escape, mispell_dict.keys())))
    return mispell_dict, mispell_re


def replace_typical_misspell(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    text = str(text)
    mispellings, mispellings_re = _get_mispell(mispell_dict)

    def replace(match):
        return mispellings[match.group(0)]

    return mispellings_re.sub(replace, text)


def clean_data(df, columns: list):
    for col in columns:
        df[col] = df[col].fillna("").astype(str)
        df[col] = df[col].apply(lambda x: clean_text(x.lower()))
        df[col] = df[col].apply(lambda x: replace_typical_misspell(x))
    return df




2026-05-12 01:35:57.134367: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 01:35:57.149167: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778549757.170262      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778549757.176369      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 01:35:57.195032: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Data load (keep paths unchanged)
df_train = pd.read_csv("/kaggle/input/google-quest-challenge/train.csv")
df_test = pd.read_csv("/kaggle/input/google-quest-challenge/test.csv")
df_submission = pd.read_csv(
    "/kaggle/input/google-quest-challenge/sample_submission.csv"
)


# Fix: make token collection local to each model run to avoid cross-run leakage/duplication
def get_words_factory():
    tokens = []

    def get_words(col):
        # Fix: robust to NaN
        if col is None or (isinstance(col, float) and np.isnan(col)):
            col = ""
        col = str(col)
        toks = []
        for x in sent_tokenize(col):
            wx = word_tokenize(x)
            tokens.extend(wx)
            toks.extend(wx)
        return toks

    return tokens, get_words


def convert_to_indx(col, word2idx, oov_index):
    return [word2idx[word] if word in word2idx else oov_index for word in col]


def LSTM_model_initial(
    df_train,
    df_test,
    df_submission,
    rnn_type="LSTM",
    embedding_size=200,
    rnn_units=64,
    maxlen_qt=26,
    maxlen_qb=260,
    maxlen_an=210,
    dropout_rate=0.2,
    dense_hidden_units=60,
    epochs=2,
):
    columns = ["question_title", "question_body", "answer"]

    # Fix: avoid SettingWithCopy + ensure clean strings
    df_train = df_train.copy()
    df_test = df_test.copy()

    df_train = clean_data(df_train, columns)
    df_test = clean_data(df_test, columns)

    tokens, get_words = get_words_factory()
    for col in columns:
        df_train[col] = df_train[col].apply(get_words)
        df_test[col] = df_test[col].apply(get_words)

    vocab = sorted(list(set(tokens)))
    vocab_size = len(vocab)

    word2idx = {word: idx for idx, word in enumerate(vocab)}
    oov_index = vocab_size  # last index reserved for OOV

    for col in columns:
        df_train[col] = df_train[col].apply(
            lambda x: convert_to_indx(x, word2idx, oov_index)
        )
        df_test[col] = df_test[col].apply(
            lambda x: convert_to_indx(x, word2idx, oov_index)
        )

    X_train_question_title = pad_sequences(
        df_train["question_title"], maxlen=maxlen_qt, padding="post", value=0
    )
    X_train_question_body = pad_sequences(
        df_train["question_body"], maxlen=maxlen_qb, padding="post", value=0
    )
    X_train_answer = pad_sequences(
        df_train["answer"], maxlen=maxlen_an, padding="post", value=0
    )

    X_test_question_title = pad_sequences(
        df_test["question_title"], maxlen=maxlen_qt, padding="post", value=0
    )
    X_test_question_body = pad_sequences(
        df_test["question_body"], maxlen=maxlen_qb, padding="post", value=0
    )
    X_test_answer = pad_sequences(
        df_test["answer"], maxlen=maxlen_an, padding="post", value=0
    )

    target_columns = df_submission.columns[1:]
    y_train = df_train[target_columns].values

    inpqt = Input(shape=(maxlen_qt,), name="inpqt")
    inpqb = Input(shape=(maxlen_qb,), name="inpqb")
    inpan = Input(shape=(maxlen_an,), name="inpan")

    # Fix: Embedding input_dim must include OOV index => vocab_size + 1
    Eqt = Embedding(vocab_size + 1, embedding_size, input_length=maxlen_qt)(inpqt)
    Eqb = Embedding(vocab_size + 1, embedding_size, input_length=maxlen_qb)(inpqb)
    Ean = Embedding(vocab_size + 1, embedding_size, input_length=maxlen_an)(inpan)

    if rnn_type == "LSTM":
        BLqt = Bidirectional(LSTM(rnn_units))(Eqt)
        BLqb = Bidirectional(LSTM(rnn_units))(Eqb)
        BLan = Bidirectional(LSTM(rnn_units))(Ean)
    else:
        BLqt = Bidirectional(GRU(rnn_units))(Eqt)
        BLqb = Bidirectional(GRU(rnn_units))(Eqb)
        BLan = Bidirectional(GRU(rnn_units))(Ean)

    Dqt = Dropout(dropout_rate)(BLqt)
    Dqb = Dropout(dropout_rate)(BLqb)
    Dan = Dropout(dropout_rate)(BLan)

    Concatenated = Concatenate()([Dqt, Dqb, Dan])
    Ds = Dense(dense_hidden_units, activation="relu")(Concatenated)
    Dsf = Dense(30, activation="sigmoid")(Ds)

    model = Model(inputs=[inpqt, inpqb, inpan], outputs=Dsf)
    model.compile("adam", "binary_crossentropy", metrics=["accuracy"])

    model.fit(
        {
            "inpqt": X_train_question_title,
            "inpqb": X_train_question_body,
            "inpan": X_train_answer,
        },
        y_train,
        batch_size=32,
        epochs=epochs,
        validation_split=0.1,
        verbose=2,
    )

    y_test = model.predict(
        {
            "inpqt": X_test_question_title,
            "inpqb": X_test_question_body,
            "inpan": X_test_answer,
        },
        verbose=0,
    )
    y_test = np.clip(y_test, 0.0, 1.0)

    # Fix: write submission with exact column order as sample_submission
    sub = pd.DataFrame(y_test, columns=target_columns)
    sub.insert(0, "qa_id", df_test["qa_id"].values)
    sub = sub[df_submission.columns]
    sub.to_csv("submission.csv", index=False)

    return sub




In [3]:
# Fix: remove TF-Hub based bert_model execution (missing dataset + protobuf/tfhub issues).
# Run the intended RNN model to produce a valid submission.csv.
_ = LSTM_model_initial(df_train, df_test, df_submission, rnn_type="LSTM", epochs=2)

print("Wrote submission.csv with shape:", pd.read_csv("submission.csv").shape)
print(pd.read_csv("submission.csv").head())

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-12 01:36:13.271780: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/2
154/154 - 36s - 237ms/step - accuracy: 8.1251e-04 - loss: 0.4427 - val_accuracy: 0.0000e+00 - val_loss: 0.3995
Epoch 2/2
154/154 - 46s - 301ms/step - accuracy: 0.0069 - loss: 0.3944 - val_accuracy: 0.0073 - val_loss: 0.3874
Wrote submission.csv with shape: (608, 31)
   qa_id  question_asker_intent_understanding  question_body_critical  \
0   6516                             0.919350                0.672785   
1   6168                             0.904871                0.587558   
2   8575                             0.944243                0.757999   
3    618                             0.937563                0.665107   
4   3471                             0.883867                0.611945   

   question_conversational  question_expect_short_answer  \
0                 0.054914                      0.806680   
1                 0.011846                      0.707911   
2                 0.178106                      0.699924   
3                 0.179510                  